# 06 - Fourier par signatures sous shifted Bergomi

Objectif: tester la partie signature-Fourier avec une grille r?duite. Les param?tres lourds du stage restent dans les fonctions Python.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np

from fourier.signature_volatility import (
    build_scale_vector,
    compute_X0_sig_fourier_rescaled,
    compute_delta_sig_fourier_rescaled,
)
from models import compute_payoff_europeen, simulate_shifted_bergomi
from signatures import compute_signatures, generate_words, word_to_index


In [ ]:
T, V0, K = 1/12, 0.0625, 1.0
J, I = 20, 256
rho, varsigma, epsilon, H = -0.7, 1.6, 1/52, 0.1
M = 3
words = generate_words(M)
word_idx = word_to_index(words)
scale = build_scale_vector(words, T/J)

S, dS, sigma, states, W_hat = simulate_shifted_bergomi(1, varsigma, V0, epsilon, H, rho, T, J, I, seed=6)


In [ ]:
try:
    sig_W = compute_signatures(W_hat, depth=M)
    scale_no_empty = np.array([scale[word_idx[w]] for w in words if len(w) > 0])
    sig_W_tilde = sig_W / scale_no_empty
    print('sig_W_tilde shape:', sig_W_tilde.shape)
except ImportError as exc:
    sig_W_tilde = None
    print(exc)


In [ ]:
if sig_W_tilde is not None:
    f_eur = lambda t, u: 1j * u
    X0_sig = compute_X0_sig_fourier_rescaled(
        rho, f_eur, T, J, M, words, word_idx, varsigma, V0, epsilon, H, scale,
        V0=V0, K=K, n_laguerre=8, N_steps=4,
    )
    delta_sig = compute_delta_sig_fourier_rescaled(
        rho, S, K, f_eur, sig_W_tilde, sigma, T, J, M, words, word_idx, varsigma, V0, epsilon, H, scale,
        V0=V0, N_steps=4, n_laguerre=8,
    )
    pnl = X0_sig + np.sum(delta_sig * dS, axis=1) - compute_payoff_europeen(S, K)
    print('X0 signature-Fourier:', X0_sig)
    print('MSE:', np.mean(pnl**2), 'Mean PnL:', np.mean(pnl))
